# 11 支持向量机 SVM

依赖安装说明：`pip install numpy matplotlib scikit-learn`

SVM 的核心思想是找一条分类边界，不只要分对，还要让边界离最近的样本尽量远。这个距离叫 margin。


## 0. 学习目标和阅读地图

SVM 的重点不是概率，而是 margin。你需要掌握：

1. 最大间隔为什么能提升泛化。
2. hinge loss 只惩罚哪些样本。
3. `C` 和 `gamma` 如何控制边界复杂度。
4. kernel trick 为什么能得到非线性边界。


## 1. 数学逻辑

线性分类器：

$$f(x)=w^Tx+b$$

SVM 希望正确分类且 margin 大。软间隔线性 SVM 常用 hinge loss：

$$L = \frac{1}{2}||w||^2 + C\sum_i \max(0, 1-y_i(w^Tx_i+b))$$

只有落在 margin 内或被分错的点会产生 hinge loss，这些点就是支持向量附近的关键样本。


## 1.1 推导拆开看：hinge loss 和 margin

对标签 `y in {-1, 1}`，margin 是：

$$m_i=y_i(w^Tx_i+b)$$

如果 `m_i >= 1`，样本不仅分对，而且离边界足够远，hinge loss 为 0：

$$\max(0, 1-m_i)=0$$

如果 `m_i < 1`，样本在间隔内或被分错，会产生损失。

这就是 SVM 的核心：它主要关心边界附近的关键点，而不是所有点都同等重要。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons
from sklearn.svm import LinearSVC, SVC
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
X, y01 = make_blobs(n_samples=220, centers=2, cluster_std=1.5, random_state=42)
y = np.where(y01 == 1, 1, -1)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)


## 1.2 线性 SVM 和 RBF SVM

线性 SVM 在原始特征空间找直线或超平面。RBF kernel 等价于把数据映射到更高维空间，再在那里找线性边界。

RBF 的 `gamma` 控制单个样本影响范围：

- `gamma` 小：影响范围大，边界平滑。
- `gamma` 大：影响范围小，边界容易变碎。


In [ ]:
# 从零实现：线性 SVM 的 SGD 版本
w = np.zeros(X_train_s.shape[1])
b = 0.0
lr = 0.01
C = 1.0

for epoch in range(80):
    for xi, yi in zip(X_train_s, y_train):
        margin = yi * (xi @ w + b)
        if margin >= 1:
            grad_w = w
            grad_b = 0.0
        else:
            grad_w = w - C * yi * xi
            grad_b = -C * yi
        w -= lr * grad_w
        b -= lr * grad_b

pred = np.where(X_test_s @ w + b >= 0, 1, -1)
print('从零线性 SVM accuracy:', round(accuracy_score(y_test, pred), 3))
print('w:', np.round(w, 3), 'b:', round(b, 3))


## 1.3 从零实现代码怎么读

SGD 版本里每个样本分两种情况：

1. `margin >= 1`：样本已经安全，只做正则化收缩 `w`。
2. `margin < 1`：样本不够安全，要用 `-C * yi * xi` 推动边界修正。

这体现了 SVM 只重点处理 margin 内样本的思想。


In [ ]:
linear = LinearSVC(C=1.0, random_state=42).fit(X_train_s, y_train)
print('LinearSVC accuracy:', round(accuracy_score(y_test, linear.predict(X_test_s)), 3))

# 非线性数据上可以用 RBF kernel
Xm, ym = make_moons(n_samples=260, noise=0.25, random_state=42)
Xm_train, Xm_test, ym_train, ym_test = train_test_split(Xm, ym, random_state=42)
rbf = SVC(kernel='rbf', C=2.0, gamma='scale').fit(Xm_train, ym_train)
print('RBF SVC accuracy:', round(accuracy_score(ym_test, rbf.predict(Xm_test)), 3))

xx, yy = np.meshgrid(np.linspace(Xm[:,0].min()-0.5, Xm[:,0].max()+0.5, 180),
                     np.linspace(Xm[:,1].min()-0.5, Xm[:,1].max()+0.5, 180))
grid = np.c_[xx.ravel(), yy.ravel()]
zz = rbf.predict(grid).reshape(xx.shape)
plt.contourf(xx, yy, zz, alpha=0.25, cmap='coolwarm')
plt.scatter(Xm_train[:,0], Xm_train[:,1], c=ym_train, cmap='coolwarm', edgecolor='k', s=24)
plt.title('RBF kernel SVM 的非线性边界')
plt.show()


In [ ]:
# 诊断：RBF SVM 的 C/gamma 小网格搜索
Cs = [0.3, 1.0, 3.0]
gammas = [0.2, 1.0, 5.0]
for C_value in Cs:
    row = []
    for gamma_value in gammas:
        m_svc = SVC(kernel='rbf', C=C_value, gamma=gamma_value).fit(Xm_train, ym_train)
        row.append(accuracy_score(ym_test, m_svc.predict(Xm_test)))
    print(f'C={C_value}:', [round(v, 3) for v in row], '  gamma columns=', gammas)


## 2.1 如何诊断 SVM

SVM 对尺度敏感，所以先检查是否标准化。然后重点调：

- `C`：错误容忍度。大 C 更想分对训练集。
- `gamma`：RBF 边界局部程度。大 gamma 更容易过拟合。

如果边界过碎，通常降低 `gamma` 或 `C`。


## 2. 常见误区

- SVM 对特征尺度敏感，通常需要标准化。
- `C` 越大越强调训练集错误，可能过拟合。
- RBF kernel 很强，但 `gamma` 太大时边界会非常碎。

## 3. 小实验

- 改 `C`，观察 margin 和错误容忍度。
- 改 RBF 的 `gamma`，观察边界复杂度。
- 对比线性 SVM 和逻辑回归。


## 5. 复习清单

- SVM 最大化 margin。
- hinge loss 只惩罚 margin 内或分错样本。
- kernel 可以得到非线性边界。
- SVM 强依赖特征缩放和超参数选择。
